In [1]:
import torch
import torchvision
import torch.nn.functional as F  # Parameterless functions, like (some) activation functions
import torchvision.datasets as datasets  # Standard datasets
import torchvision.transforms as transforms  # Transformations we can perform on our dataset for augmentation
from torch import optim  # For optimizers like SGD, Adam, etc.
from torch import nn  # All neural network modules
from torch.utils.data import DataLoader
from tqdm import tqdm  # For nice progress bar!
import matplotlib.pyplot as plt
import torch.nn.functional as F

In [7]:
trans_train = transforms.Compose([ transforms.Resize(150),
                                  transforms.RandomRotation(30),
                                  transforms.RandomResizedCrop(32),
                                  transforms.RandomHorizontalFlip(),
                                  transforms.ToTensor()])


data_dir = "D:\Machine Learning\Datasets\Datasets\SheepFaceImages"
train_dataset = datasets.ImageFolder(data_dir , transform=trans_train)
test_dataset = datasets.ImageFolder(data_dir , transform=trans_valid)

batch_size = 1
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle= True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle =False)
type(train_dataset)

torchvision.datasets.folder.ImageFolder

In [19]:
import torch
import torch.nn as nn
import torchvision.models as models

# Load model
model = models.resnet18(weights='DEFAULT')

# Remove global average pooling
model.avgpool = nn.Identity()

# Replace the final layer (ResNet uses .fc, not .classifier)
model.fc = nn.Sequential(
    nn.Linear(512 * 7 * 7, 100),    # ← FIXED: 25088 input features
    nn.ReLU(),
    nn.Linear(100, 50),
    nn.ReLU(),
    nn.Linear(50, num_classes)
)

model = model.to(device)

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = 2
model = torchvision.models.resnet18(weights='DEFAULT')
model.avgpool = nn.Identity()
model.classifier = nn.Sequential( nn.Linear(25088,100), nn.ReLU(), nn.Linear(100,50), nn.ReLU(), nn.Linear(50,num_classes))
model.to(device)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [18]:
from torchsummary import summary
summary(model,(3,224,224))

RuntimeError: mat1 and mat2 shapes cannot be multiplied (2x25088 and 512x1000)

In [23]:
#%%
import torch
import torchvision
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchsummary import summary
from torch.utils.data import DataLoader
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from tqdm import tqdm
#%%
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = 2
model = torchvision.models.resnet50(weights='IMAGENET1K_V1')
# model.avgpool = nn.AdaptiveAvgPool2d((1, 1))
model.fc = nn.Sequential(
    nn.Linear(2048, 100),
    nn.ReLU(),
    nn.Linear(100, 50),
    nn.ReLU(),
    nn.Linear(50, num_classes))
#%%
summary(model, (3, 224, 224))
#%%
in_channels = 3
learning_rate = 3e-4
batch_size = 16
num_epochs = 5
#%%
trans_train = transforms.Compose([transforms.Resize(150),
                                 transforms.RandomRotation(30),
                                 transforms.RandomResizedCrop(80),
                                 transforms.RandomHorizontalFlip(),
                                 transforms.ToTensor()])


data_dir = "D:\Machine Learning\Datasets\Datasets\SheepFaceImages"
train_dataset = datasets.ImageFolder(data_dir, transform=trans_train)
test_dataset = datasets.ImageFolder(data_dir, transform=trans_valid)
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset = test_dataset, batch_size=batch_size, shuffle=True)

#%%
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adamax(model.parameters(), lr=learning_rate)

#%%
for epoch in range(num_epochs):
    losses = []
    for batch_idx, (data, targets) in enumerate(tqdm(train_loader)):
        data = data.to(device=device)
        targets = targets.to(device=device)
        scores = model(data)
        loss = criterion(scores, targets)
        losses.append(loss.item())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Cost at epoch {epoch} is {sum(losses) / len(losses):.5f}")
#%%
def check_accuracy(loader, model):
    num_correct = 0
    num_samples = 0
    model.eval()  # Set model to evaluation mode

    with torch.no_grad():  # No need to track gradients during accuracy check
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            scores = model(x)
            _, predictions = scores.max(1)
            num_correct += (predictions == y).sum()
            num_samples += predictions.size(0)

    model.train()  # Set model back to training mode

    # CRITICAL: You must return the value!
    return float(num_correct) / float(num_samples)
#%%

print(f"Accuracy on training set: {check_accuracy(train_loader, model) * 100:.2f}")
print(f"Accuracy on test set: {check_accuracy(test_loader, model) * 100:.2f}")
#%%

#%%


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to C:\Users\dbda.STUDENTSDC/.cache\torch\hub\checkpoints\resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 52.1MB/s]


----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 112, 112]           9,408
       BatchNorm2d-2         [-1, 64, 112, 112]             128
              ReLU-3         [-1, 64, 112, 112]               0
         MaxPool2d-4           [-1, 64, 56, 56]               0
            Conv2d-5           [-1, 64, 56, 56]           4,096
       BatchNorm2d-6           [-1, 64, 56, 56]             128
              ReLU-7           [-1, 64, 56, 56]               0
            Conv2d-8           [-1, 64, 56, 56]          36,864
       BatchNorm2d-9           [-1, 64, 56, 56]             128
             ReLU-10           [-1, 64, 56, 56]               0
           Conv2d-11          [-1, 256, 56, 56]          16,384
      BatchNorm2d-12          [-1, 256, 56, 56]             512
           Conv2d-13          [-1, 256, 56, 56]          16,384
      BatchNorm2d-14          [-1, 256,

  0%|          | 0/105 [00:00<?, ?it/s]


IndexError: Target 2 is out of bounds.

In [25]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
from torchsummary import summary

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = 2  # Assuming binary classification (e.g., healthy vs sick sheep)

# Load pre-trained ResNet50
model = torchvision.models.resnet50(weights='IMAGENET1K_V1')

# Replace the final fully connected layer (DO NOT remove avgpool!)
# ResNet50 after avgpool outputs 2048 features
model.fc = nn.Sequential(
    nn.Linear(2048, 100),
    nn.ReLU(),
    nn.Linear(100, 50),
    nn.ReLU(),
    nn.Linear(50, num_classes)
)

model = model.to(device)

# Print model summary to verify
summary(model, (3, 224, 224))

# Hyperparameters
learning_rate = 3e-4
batch_size = 16
num_epochs = 5

# Proper transforms for ResNet50 (224×224 input + ImageNet normalization)
trans_train = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

trans_valid = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Data loading – IMPORTANT: separate train & val folders!
data_dir = r"D:\Machine Learning\Datasets\Datasets\SheepFaceImages"

# Assuming folder structure: SheepFaceImages/train and SheepFaceImages/val
train_dataset = torchvision.datasets.ImageFolder(data_dir, transform=trans_train)
test_dataset  = torchvision.datasets.ImageFolder(data_dir,transform=trans_valid)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=4)

# Loss and optimizer (Adamax is fine, but Adam or AdamW often works better)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)  # Changed to Adam for simplicity

# Training loop with progress bar
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    loop = tqdm(train_loader, leave=True)

    for data, targets in loop:
        data = data.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        scores = model(data)
        loss = criterion(scores, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        loop.set_description(f"Epoch [{epoch+1}/{num_epochs}]")
        loop.set_postfix(loss=loss.item())

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs} - Average Loss: {avg_loss:.5f}")

# Accuracy function
def check_accuracy(loader, model):
    model.eval()
    num_correct = 0
    num_samples = 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            scores = model(x)
            _, predictions = scores.max(1)
            num_correct += (predictions == y).sum()
            num_samples += y.size(0)
    model.train()  # restore training mode
    return num_correct / num_samples

# Print accuracies
train_acc = check_accuracy(train_loader, model)
test_acc  = check_accuracy(test_loader, model)

print(f"Accuracy on training set: {train_acc * 100:.2f}%")
print(f"Accuracy on test set:  {test_acc * 100:.2f}%")

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 112, 112]           9,408
       BatchNorm2d-2         [-1, 64, 112, 112]             128
              ReLU-3         [-1, 64, 112, 112]               0
         MaxPool2d-4           [-1, 64, 56, 56]               0
            Conv2d-5           [-1, 64, 56, 56]           4,096
       BatchNorm2d-6           [-1, 64, 56, 56]             128
              ReLU-7           [-1, 64, 56, 56]               0
            Conv2d-8           [-1, 64, 56, 56]          36,864
       BatchNorm2d-9           [-1, 64, 56, 56]             128
             ReLU-10           [-1, 64, 56, 56]               0
           Conv2d-11          [-1, 256, 56, 56]          16,384
      BatchNorm2d-12          [-1, 256, 56, 56]             512
           Conv2d-13          [-1, 256, 56, 56]          16,384
      BatchNorm2d-14          [-1, 256,

  0%|          | 0/105 [00:15<?, ?it/s]


IndexError: Target 2 is out of bounds.